# SkinAge Atlas: Cross-Cohort Transcriptomic Analysis of Human Skin Ageing & Independent-Cohort Biomarker Validation

**MSc Bioinformatics Portfolio Project** | High-Throughput Transcriptomics & Machine Learning (Nordic / European Biotech & Health-Tech)

---
### Research Questions
1. **Chronological Ageing:** Which genes and functional pathways change significantly with chronological age in human skin?
2. **Extrinsic Photoaging:** Do age-associated genes differ between sun-exposed and sun-protected skin?
3. **Cross-Cohort Generalization:** Can a machine learning model (Elastic Net) trained on RNA-seq primary fibroblasts predict chronological age in completely separate in vivo human skin tissue?

---
### Strict Scientific Principles
* **Zero Data Fabrication:** Every accession number, platform, sample count, and p-value is grounded in live NCBI GEO data.
* **Zero Data Leakage:** Independent test cohort (GSE38308) is completely withheld from training, feature pre-selection, and scaler parameter fitting.
* **Association Disclaimer:** All reported findings are transcriptomic associations, not causal claims or cosmetic product endorsements.


## 1. Setup & Environment Initialization
Install pinned dependencies and import core bioinformatics and data science packages.


In [ ]:
# Run in Google Colab
# !pip install -q numpy pandas scipy statsmodels scikit-learn matplotlib seaborn plotly gseapy openpyxl requests

import os, sys, ssl, gzip, tarfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score
import gseapy as gp

# Styling
sns.set_theme(style="ticks", font_scale=1.05)
plt.rcParams["font.sans-serif"] = "Helvetica, Arial, DejaVu Sans"

for folder in ["data/raw", "data/processed", "data/metadata", "results/figures", "results/tables"]:
    os.makedirs(folder, exist_ok=True)
print("Directories initialized successfully.")


## 2. Programmatic Data Download (GEO Ingestion)
Downloads NCBI  for Ensembl-to-HGNC mapping, GSE226189 Discovery cohort (RNA-seq, N=82), and GSE38308 Validation cohort (Microarray, N=42).


In [ ]:
import src.download as dl

ens_map = dl.build_ensembl_symbol_map()
meta_disc, counts_disc = dl.download_gse226189()
meta_val, expr_val = dl.download_gse38308()
print(f"Discovery cohort loaded: {counts_disc.shape[0]:,} genes across {counts_disc.shape[1]} samples")
print(f"Validation cohort loaded: {expr_val.shape[0]:,} genes across {expr_val.shape[1]} samples")


## 3. Quality Control, Normalization & PCA
Filters low-abundance genes (count >= 10 in >= 20% of samples), performs log2-CPM normalization, evaluates sample-to-sample correlations, and plots PCA.


In [ ]:
import src.qc as qc

meta_disc, filt_counts, log2_cpm, pca_disc = qc.run_gse226189_qc()
meta_val, expr_val, pca_val = qc.run_gse38308_qc()


## 4. Differential Expression & Association Modeling
1. **GSE226189 (Chronological Ageing):** Linear regression  across all 82 samples and Young vs Old comparison.
2. **GSE38308 (Photoaging):** Intra-individual paired t-test for Sun-Exposed vs Sun-Protected facial skin across 21 matched donors.


In [ ]:
import src.de as de

de_disc, de_yo = de.run_gse226189_de()
de_val = de.run_gse38308_de()


## 5. Functional Pathway Enrichment Analysis (GSEAPy PreRank)
PreRanked GSEA across MSigDB Hallmark, Reactome, and GO Biological Process gene sets.


In [ ]:
import src.pathway as pw

hallmark_age, reactome_age, hallmark_sun = pw.run_pathway_analysis()


## 6. Transcriptomic Age Predictor (Elastic Net)
Trains Elastic Net regression with 5-fold cross-validation on GSE226189, and evaluates directly on independent cohort GSE38308 without refitting.


In [ ]:
import src.model as ml

metrics_df, coef_df = ml.run_age_predictor()
print(metrics_df)
